In [1]:
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import imblearn

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Workers")

df = pd.read_csv('../data/cleaned.csv')
prep = joblib.load('../models/clustering/preprocessor.pkl')
kmeans = joblib.load('../models/clustering/kmeans_churn_model.pkl')

x = df.drop('Churn', axis=1)
y = df['Churn']

le = LabelEncoder()
y = le.fit_transform(y)

In [3]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

x_train_prep = prep.transform(x_train)
x_test_prep = prep.transform(x_test)

x_train_scaled = pd.DataFrame(x_train_prep, index=x_train.index)
x_test_scaled = pd.DataFrame(x_test_prep, index=x_test.index)

x_train_scaled['cluster'] = kmeans.predict(x_train_prep)
x_test_scaled['cluster'] = kmeans.predict(x_test_prep)

x_train_scaled.columns = x_train_scaled.columns.astype(str)
x_test_scaled.columns = x_test_scaled.columns.astype(str)

x_train_scaled = x_train_scaled.astype(float)
x_test_scaled = x_test_scaled.astype(float)

In [4]:
models = {
    'Logistic_Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'model__C': [0.1, 1.0, 10.0],
            'model__solver': ['lbfgs', 'liblinear']
        }
    },
    'Decision_Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'model__max_depth': [3, 5, 10, None],
            'model__min_samples_split': [2, 5, 10]
        }
    },
    'Random_Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [5, 10, None]
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'model__C': [0.1, 1, 10],
            'model__kernel': ['rbf', 'linear']
        }
    },
    'Gradient_Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'model__n_estimators': [50, 100],
            'model__learning_rate': [0.01, 0.1, 0.2]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 6],
            'model__learning_rate': [0.01, 0.1]
        }
    }
}

In [5]:
res =[]

for name,config in models.items():
    print(f"--- Running MLflow Run: {name} ---")
    with mlflow.start_run(run_name=name):
        
        pipeline = Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('model', config['model'])
        ])
        
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=config['params'],
            scoring='f1',
            cv=5,
            n_jobs=-1
        )
        
        grid.fit(x_train_scaled, y_train)
        best_model = grid.best_estimator_
        
        y_pred = best_model.predict(x_test_scaled)
        y_proba = best_model.predict_proba(x_test_scaled)[:, 1]
        
        metrics = {
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1_score': f1_score(y_test, y_pred),
            'roc_auc': roc_auc_score(y_test, y_proba)
        }
        
        mlflow.log_params(grid.best_params_)
        mlflow.log_metrics(metrics)
        mlflow.log_param("model_type", name)
        
        input_example = x_train_scaled.head(5)
        try:
            mlflow.sklearn.log_model(
            sk_model=best_model,
            name="model",
            input_example=input_example,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE
            )
            print(f" Modèle {name} sauvegardé avec succès dans MLflow !")
        except Exception as e:
            print(f" Erreur lors de la sauvegarde du modèle {name} : {e}")
        
        res.append({
            'Modèle': name,
            'run_id':mlflow.active_run().info.run_id,
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1-Score': metrics['f1_score'],
            'ROC-AUC': metrics['roc_auc'],
            **grid.best_params_
        })

--- Running MLflow Run: Logistic_Regression ---


2026/10/09 14:57:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missi

 Modèle Logistic_Regression sauvegardé avec succès dans MLflow !
--- Running MLflow Run: Decision_Tree ---


2026/10/09 14:57:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missi

 Modèle Decision_Tree sauvegardé avec succès dans MLflow !
--- Running MLflow Run: Random_Forest ---


...

 Modèle SVM sauvegardé avec succès dans MLflow !
--- Running MLflow Run: Gradient_Boosting ---


2026/10/09 14:58:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missi

 Modèle Gradient_Boosting sauvegardé avec succès dans MLflow !
--- Running MLflow Run: XGBoost ---


2026/10/09 14:58:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missi

 Modèle XGBoost sauvegardé avec succès dans MLflow !


In [6]:
df_results = pd.DataFrame(res).sort_values(by='Recall', ascending=False)
display(df_results[['Modèle', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']])

,Modèle,Precision,Recall,F1-Score,ROC-AUC
0,Logistic_Regression,0.507666,0.796791,0.620187,0.840399
3,SVM,0.507853,0.778075,0.614572,0.835418
2,Random_Forest,0.526412,0.772727,0.626219,0.842769
5,XGBoost,0.551456,0.759358,0.638920,0.842340
4,Gradient_Boosting,0.550201,0.732620,0.628440,0.842325
1,Decision_Tree,0.552632,0.673797,0.607229,0.820144


In [ ]:
best_row = df_results.iloc[0]
best_run_id = best_row['run_id']

print("Meilleur modèle :", best_row['Modèle'])
print("Run ID :", best_run_id)

model_uri = f"runs:/{best_run_id}/model"
print("Model URI :", model_uri)

Meilleur modèle : Logistic_Regression
Run ID : e08c8750b8244921985c8d8c13e5e32e
Model URI : runs:/e08c8750b8244921985c8d8c13e5e32e/model


In [10]:
model_uri = f"runs:/{best_run_id}/model"

best_model = mlflow.sklearn.load_model(model_uri)

# Enregistrer le modèle localement
joblib.dump(best_model, "../models/model.pkl")

print("Modèle exporté avec succès !")

Modèle exporté avec succès !
